# Kapitola 7: Používání příkladů (Few-Shot Prompting)

- [Lekce](#lesson)
- [Cvičení](#exercises)
- [Příkladové hřiště](#example-playground)

## Nastavení

Spusťte následující buňku pro nastavení, abyste načetli svůj API klíč a vytvořili pomocnou funkci `get_completion`.

In [ ]:
```python
# Import vestavěnou knihovnu regulárních výrazů v Pythonu
import re
import boto3
import json

# Import modulu hints z balíčku utils
import os
import sys
module_path = ".."
sys.path.append(os.path.abspath(module_path))
from utils import hints

# Načtení proměnné MODEL_NAME z úložiště IPython
%store -r MODEL_NAME
%store -r AWS_REGION

client = boto3.client('bedrock-runtime', region_name=AWS_REGION)

def get_completion(prompt, system='', prefill=''):
    body = json.dumps(
        {
            "anthropic_version": '',
            "max_tokens": 2000,
            "messages":[
              {"role": "user", "content": prompt},
              {"role": "assistant", "content": prefill}
            ],
            "temperature": 0.0,
            "top_p": 1,
            "system": system
        }
    )
    response = client.invoke_model(body=body, modelId=MODEL_NAME)
    response_body = json.loads(response.get('body').read())

    return response_body.get('content')[0].get('text')
```

---

## Lekce

**Poskytnutí příkladů Claudovi, jak chcete, aby se choval (nebo jak nechcete, aby se choval), je mimořádně efektivní** pro:
- Získání správné odpovědi
- Získání odpovědi ve správném formátu

Tento druh promptingu se také nazývá "**few shot prompting**". Můžete se také setkat s frázemi "zero-shot", "n-shot" nebo "one-shot". Počet "shots" označuje, kolik příkladů je použito v promptu.

### Příklady

Představte si, že jste vývojář, který se snaží vytvořit "rodičovského bota", který odpovídá na otázky dětí. **Claudeova výchozí odpověď je poměrně formální a robotická**. To by zlomilo srdce dítěte.

In [ ]:
```python
# Prompt
PROMPT = "Will Santa bring me presents on Christmas?"

# Vytisknout odpověď od Claude
print(get_completion(PROMPT))
```

Můžete si dát čas na popis požadovaného tónu, ale je mnohem jednodušší **dát Claudovi několik příkladů ideálních odpovědí**.

In [ ]:
```python
# Prompt
PROMPT = """Please complete the conversation by writing the next line, speaking as "A".
Q: Is the tooth fairy real?
A: Of course, sweetie. Wrap up your tooth and put it under your pillow tonight. There might be something waiting for you in the morning.
Q: Will Santa bring me presents on Christmas?"""

# Vytisknout odpověď od Claude
print(get_completion(PROMPT))
```

V následujícím příkladu formátování bychom mohli Clauda krok za krokem provést sadou instrukcí, jak extrahovat jména a profese a poté je formátovat přesně tak, jak chceme, nebo bychom mohli **poskytnout Claudovi několik správně naformátovaných příkladů a Claude může z toho extrapolovat**. Všimněte si `<individuals>` v části `assistant`, aby Claude začal správně.

In [ ]:
# Prompt template with a placeholder for the variable content
PROMPT = """Silvermist Hollow, a charming village, was home to an extraordinary group of individuals.
Among them was Dr. Liam Patel, a neurosurgeon who revolutionized surgical techniques at the regional medical center.
Olivia Chen was an innovative architect who transformed the village's landscape with her sustainable and breathtaking designs.
The local theater was graced by the enchanting symphonies of Ethan Kovacs, a professionally-trained musician and composer.
Isabella Torres, a self-taught chef with a passion for locally sourced ingredients, created a culinary sensation with her farm-to-table restaurant, which became a must-visit destination for food lovers.
These remarkable individuals, each with their distinct talents, contributed to the vibrant tapestry of life in Silvermist Hollow.
<individuals>
1. Dr. Liam Patel [NEUROSURGEON]
2. Olivia Chen [ARCHITECT]
3. Ethan Kovacs [MISICIAN AND COMPOSER]
4. Isabella Torres [CHEF]
</individuals>

At the heart of the town, Chef Oliver Hamilton has transformed the culinary scene with his farm-to-table restaurant, Green Plate. Oliver's dedication to sourcing local, organic ingredients has earned the establishment rave reviews from food critics and locals alike.
Just down the street, you'll find the Riverside Grove Library, where head librarian Elizabeth Chen has worked diligently to create a welcoming and inclusive space for all. Her efforts to expand the library's offerings and establish reading programs for children have had a significant impact on the town's literacy rates.
As you stroll through the charming town square, you'll be captivated by the beautiful murals adorning the walls. These masterpieces are the work of renowned artist, Isabella Torres, whose talent for capturing the essence of Riverside Grove has brought the town to life.
Riverside Grove's athletic achievements are also worth noting, thanks to former Olympic swimmer-turned-coach, Marcus Jenkins. Marcus has used his experience and passion to train the town's youth, leading the Riverside Grove Swim Team to several regional championships.
<individuals>
1. Oliver Hamilton [CHEF]
2. Elizabeth Chen [LIBRARIAN]
3. Isabella Torres [ARTIST]
4. Marcus Jenkins [COACH]
</individuals>

Oak Valley, a charming small town, is home to a remarkable trio of individuals whose skills and dedication have left a lasting impact on the community.
At the town's bustling farmer's market, you'll find Laura Simmons, a passionate organic farmer known for her delicious and sustainably grown produce. Her dedication to promoting healthy eating has inspired the town to embrace a more eco-conscious lifestyle.
In Oak Valley's community center, Kevin Alvarez, a skilled dance instructor, has brought the joy of movement to people of all ages. His inclusive dance classes have fostered a sense of unity and self-expression among residents, enriching the local arts scene.
Lastly, Rachel O'Connor, a tireless volunteer, dedicates her time to various charitable initiatives. Her commitment to improving the lives of others has been instrumental in creating a strong sense of community within Oak Valley.
Through their unique talents and unwavering dedication, Laura, Kevin, and Rachel have woven themselves into the fabric of Oak Valley, helping to create a vibrant and thriving small town."""

# Prefill for Claude's response
PREFILL = "<individuals>"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN:")
print(PROMPT)
print("\nASSISTANT TURN:")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

Pokud byste chtěli experimentovat s prompty lekce, aniž byste měnili jakýkoli obsah výše, přejděte až na konec poznámkového bloku lekce a navštivte [**Example Playground**](#example-playground).

## Cvičení
- [Cvičení 7.1 - Formátování e-mailů pomocí příkladů](#exercise-71---email-formatting-via-examples)

### Cvičení 7.1 - Formátování e-mailů pomocí příkladů
Budeme znovu provádět Cvičení 6.2, ale tentokrát upravíme `PROMPT` tak, aby používal příklady e-mailů ve stylu "few-shot" + správnou klasifikaci (a formátování), abychom přiměli Claudeho k výstupu správné odpovědi. Chceme, aby *poslední* písmeno výstupu od Claudeho bylo písmeno kategorie.

Pokud zapomenete, která písmenková kategorie je správná pro každý e-mail, podívejte se na komentáře vedle každého e-mailu v seznamu `EMAILS`.

Pamatujte, že toto jsou kategorie pro e-maily:										
- (A) Dotaz před prodejem
- (B) Rozbitá nebo vadná položka
- (C) Dotaz na fakturaci
- (D) Jiné (prosím vysvětlete)								

In [ ]:
```python
# Šablona promptu s zástupným symbolem pro proměnnou obsah
PROMPT = """Please classify this email as either green or blue: {email}"""

# Předvyplnění pro odpověď Claude
PREFILL = ""

# Obsah proměnné uložený jako seznam
EMAILS = [
    "Hi -- My Mixmaster4000 is producing a strange noise when I operate it. It also smells a bit smoky and plasticky, like burning electronics.  I need a replacement.", # (B) Rozbitá nebo vadná položka
    "Can I use my Mixmaster 4000 to mix paint, or is it only meant for mixing food?", # (A) Dotaz před nákupem NEBO (D) Jiné (prosím vysvětlete)
    "I HAVE BEEN WAITING 4 MONTHS FOR MY MONTHLY CHARGES TO END AFTER CANCELLING!!  WTF IS GOING ON???", # (C) Dotaz na fakturaci
    "How did I get here I am not good with computer.  Halp." # (D) Jiné (prosím vysvětlete)
]

# Správné kategorizace uložené jako seznam seznamů pro možnost více správných kategorizací na email
ANSWERS = [
    ["B"],
    ["A","D"],
    ["C"],
    ["D"]
]

# Iterace přes seznam emailů
for i,email in enumerate(EMAILS):
    
    # Nahrazení textu emailu do proměnné zástupného symbolu emailu
    formatted_prompt = PROMPT.format(email=email)
   
    # Získání odpovědi od Claude
    response = get_completion(formatted_prompt, prefill=PREFILL)

    # Hodnocení odpovědi od Claude
    grade = any([bool(re.search(ans, response[-1])) for ans in ANSWERS[i]])
    
    # Tisk odpovědi od Claude
    print("--------------------------- Full prompt with variable substutions ---------------------------")
    print("USER TURN")
    print(formatted_prompt)
    print("\nASSISTANT TURN")
    print(PREFILL)
    print("\n------------------------------------- Claude's response -------------------------------------")
    print(response)
    print("\n------------------------------------------ GRADING ------------------------------------------")
    print("This exercise has been correctly solved:", grade, "\n\n\n\n\n\n")
```

❓ Pokud chcete nápovědu, spusťte buňku níže!

In [ ]:
print(hints.exercise_7_1_hint)

Stále máte potíže? Spusťte buňku níže pro příklad řešení.

In [ ]:
print(hints.exercise_7_1_solution)

### Gratulujeme!

Pokud jste vyřešili všechny úkoly až do tohoto bodu, jste připraveni přejít k další kapitole. Šťastné promptování!

---

## Příklad hřiště

Toto je prostor, kde můžete volně experimentovat s příklady promptů uvedenými v této lekci a upravovat prompty, abyste viděli, jak to může ovlivnit odpovědi Claude.

In [ ]:
```python
# Prompt
PROMPT = "Will Santa bring me presents on Christmas?"

# Vytiskne odpověď od Claude
print(get_completion(PROMPT))
```

In [ ]:
```python
# Prompt
PROMPT = """Please complete the conversation by writing the next line, speaking as "A".
Q: Is the tooth fairy real?
A: Of course, sweetie. Wrap up your tooth and put it under your pillow tonight. There might be something waiting for you in the morning.
Q: Will Santa bring me presents on Christmas?"""

# Vytisknout odpověď od Claude
print(get_completion(PROMPT))
```

In [ ]:
# Prompt template with a placeholder for the variable content
PROMPT = """Silvermist Hollow, a charming village, was home to an extraordinary group of individuals.
Among them was Dr. Liam Patel, a neurosurgeon who revolutionized surgical techniques at the regional medical center.
Olivia Chen was an innovative architect who transformed the village's landscape with her sustainable and breathtaking designs.
The local theater was graced by the enchanting symphonies of Ethan Kovacs, a professionally-trained musician and composer.
Isabella Torres, a self-taught chef with a passion for locally sourced ingredients, created a culinary sensation with her farm-to-table restaurant, which became a must-visit destination for food lovers.
These remarkable individuals, each with their distinct talents, contributed to the vibrant tapestry of life in Silvermist Hollow.
<individuals>
1. Dr. Liam Patel [NEUROSURGEON]
2. Olivia Chen [ARCHITECT]
3. Ethan Kovacs [MISICIAN AND COMPOSER]
4. Isabella Torres [CHEF]
</individuals>

At the heart of the town, Chef Oliver Hamilton has transformed the culinary scene with his farm-to-table restaurant, Green Plate. Oliver's dedication to sourcing local, organic ingredients has earned the establishment rave reviews from food critics and locals alike.
Just down the street, you'll find the Riverside Grove Library, where head librarian Elizabeth Chen has worked diligently to create a welcoming and inclusive space for all. Her efforts to expand the library's offerings and establish reading programs for children have had a significant impact on the town's literacy rates.
As you stroll through the charming town square, you'll be captivated by the beautiful murals adorning the walls. These masterpieces are the work of renowned artist, Isabella Torres, whose talent for capturing the essence of Riverside Grove has brought the town to life.
Riverside Grove's athletic achievements are also worth noting, thanks to former Olympic swimmer-turned-coach, Marcus Jenkins. Marcus has used his experience and passion to train the town's youth, leading the Riverside Grove Swim Team to several regional championships.
<individuals>
1. Oliver Hamilton [CHEF]
2. Elizabeth Chen [LIBRARIAN]
3. Isabella Torres [ARTIST]
4. Marcus Jenkins [COACH]
</individuals>

Oak Valley, a charming small town, is home to a remarkable trio of individuals whose skills and dedication have left a lasting impact on the community.
At the town's bustling farmer's market, you'll find Laura Simmons, a passionate organic farmer known for her delicious and sustainably grown produce. Her dedication to promoting healthy eating has inspired the town to embrace a more eco-conscious lifestyle.
In Oak Valley's community center, Kevin Alvarez, a skilled dance instructor, has brought the joy of movement to people of all ages. His inclusive dance classes have fostered a sense of unity and self-expression among residents, enriching the local arts scene.
Lastly, Rachel O'Connor, a tireless volunteer, dedicates her time to various charitable initiatives. Her commitment to improving the lives of others has been instrumental in creating a strong sense of community within Oak Valley.
Through their unique talents and unwavering dedication, Laura, Kevin, and Rachel have woven themselves into the fabric of Oak Valley, helping to create a vibrant and thriving small town."""

# Prefill for Claude's response
PREFILL = "<individuals>"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN:")
print(PROMPT)
print("\nASSISTANT TURN:")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))